## Mlflow

In [1]:
!pip -q install -U mlflow pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.1 MB/s eta 0:00:00


In [2]:
import os
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

import mlflow
import mlflow.sklearn

from pyngrok import ngrok

In [3]:
def saca_metricas(y_true, y_pred):
    """Imprime matriz de confusión y accuracy."""
    print('Matriz de Confusión')
    print(confusion_matrix(y_true, y_pred))
    print('Accuracy:', accuracy_score(y_true, y_pred))


In [4]:
# Load dataset (adjust the path if needed)
candidates = [
    pathlib.Path('../bank-full.csv'),
    pathlib.Path('/content/bank-full.csv'),
    pathlib.Path('bank-full.csv'),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('No encuentro bank-full.csv. Sube el archivo a Colab o ajusta la ruta en esta celda.')

df_bank = pd.read_csv(data_path, sep=';')
df_bank.head()


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [5]:
categorical_feature = df_bank.select_dtypes(include='object').columns
numerical_feature = df_bank.select_dtypes(exclude='object').columns

In [6]:
for k in categorical_feature:
    le = LabelEncoder()
    df_bank[k] = le.fit_transform(df_bank[k])

In [7]:
X_train, X_test, y_train, y_test = train_test_split(df_bank.drop('y',axis=1),
                                                    df_bank.y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify = df_bank.y)

In [8]:
df_bank

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,4,1,2,0,2143,1,0,2,5,8,261,1,-1,0,3,0
1,44,9,2,1,0,29,1,0,2,5,8,151,1,-1,0,3,0
2,33,2,1,1,0,2,1,1,2,5,8,76,1,-1,0,3,0
3,47,1,1,3,0,1506,1,0,2,5,8,92,1,-1,0,3,0
4,33,11,2,3,0,1,0,0,2,5,8,198,1,-1,0,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,9,1,2,0,825,0,0,0,17,9,977,3,-1,0,3,1
45207,71,5,0,0,0,1729,0,0,0,17,9,456,2,-1,0,3,1
45208,72,5,1,1,0,5715,0,0,0,17,9,1127,5,184,3,2,1
45209,57,1,1,1,0,668,0,0,1,17,9,508,4,-1,0,3,0


In [9]:
n_estimators = 20
semilla = 42
model1 = RandomForestClassifier(n_estimators=n_estimators, random_state=semilla)

### Mlflow

In [10]:
TRACKING_DIR = '/content/mlruns'
os.makedirs(TRACKING_DIR, exist_ok=True)
mlflow.set_tracking_uri(f'file:{TRACKING_DIR}')
mlflow.set_experiment('ejercicio_banco_mlflow')
print('tracking_uri =', mlflow.get_tracking_uri())


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/14 09:03:41 INFO mlflow.tracking.fluent: Experiment with name 'ejercicio_banco_mlflow' does not exist. Creating a new experiment.


tracking_uri = file:/content/mlruns


In [11]:
def run_and_log(model, run_name, X_train, y_train, X_test, y_test):
    with mlflow.start_run(run_name=run_name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        for k, v in model.get_params().items():
            mlflow.log_param(k, v)

        mlflow.log_metric('accuracy', float(acc))

        mlflow.sklearn.log_model(model, 'model')

        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d')
        plt.title(f'Confusion matrix - {run_name}')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.tight_layout()
        cm_path = f'confusion_matrix_{run_name}.png'
        plt.savefig(cm_path, dpi=150)
        plt.close()
        mlflow.log_artifact(cm_path)

        print(run_name, 'accuracy =', acc)
        return acc


In [12]:
model_untuned = RandomForestClassifier(n_estimators=20, random_state=semilla)
run_and_log(model_untuned, 'untuned_random_forest', X_train, y_train, X_test, y_test)

2026/03/14 09:03:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 09:03:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


untuned_random_forest accuracy = 0.9009178370009953


0.9009178370009953

In [15]:
with open("ngrok_token.txt", "r") as f:
    ngrok_token = f.read().strip()

In [16]:
!pkill -f "mlflow (ui|server)" || true

In [17]:
!nohup mlflow server \
  --backend-store-uri file:///content/mlruns \
  --host 0.0.0.0 \
  --port 5000 \
  --allowed-hosts "*" \
  --cors-allowed-origins "*" \
  > /content/mlflow_server.log 2>&1 &

In [18]:
!tail -n 80 /content/mlflow_server.log

In [19]:
ngrok.kill()
ngrok.set_auth_token(ngrok_token)
t = ngrok.connect("127.0.0.1:5000", "http", host_header="rewrite")
print(t.public_url)

https://karina-unpleaded-amatively.ngrok-free.dev


### Si hacemos un pequeño cambio/finetuning...

In [20]:
n_estimators = 200 ## subimos los árboles
semilla = 42
model_tuned = RandomForestClassifier(n_estimators=n_estimators, random_state=semilla)

In [21]:
run_and_log(model_tuned, 'tuned_random_forest', X_train, y_train, X_test, y_test)

2026/03/14 09:06:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 09:06:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


tuned_random_forest accuracy = 0.9053411478491651


0.9053411478491651

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
